# Agriculture model training-only notebook

This file is derived from the uploaded notebook. It keeps the final in-memory NASA POWER -> feature engineering -> target generation pipeline and the final strict-feature LightGBM production training cell.

The final export is written to `models/agriculture/agriculture.pkl` for the Flask repo.


In [1]:
# ==============================================================================
# ZERO-DISK-DEPENDENCY END-TO-END PIPELINE: INGEST -> FEATURES -> TARGETS
# ==============================================================================
import time
import requests
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# 1. 28 Representative State & Zone Anchor Districts Across India
INDIAN_GRID_28 = [
    {"district": "Ludhiana", "state": "Punjab", "latitude": 30.9010, "longitude": 75.8573, "zone": "Trans-Gangetic"},
    {"district": "Karnal", "state": "Haryana", "latitude": 29.6857, "longitude": 76.9905, "zone": "Trans-Gangetic"},
    {"district": "Meerut", "state": "Uttar Pradesh", "latitude": 28.9845, "longitude": 77.7064, "zone": "Upper Gangetic"},
    {"district": "Varanasi", "state": "Uttar Pradesh", "latitude": 25.3176, "longitude": 82.9739, "zone": "Middle Gangetic"},
    {"district": "Shimla", "state": "Himachal Pradesh", "latitude": 31.1048, "longitude": 77.1734, "zone": "Western Himalayan"},
    {"district": "Dehradun", "state": "Uttarakhand", "latitude": 30.3165, "longitude": 78.0322, "zone": "Western Himalayan"},
    {"district": "Srinagar", "state": "Jammu & Kashmir", "latitude": 34.0837, "longitude": 74.7973, "zone": "Western Himalayan"},
    {"district": "Patna", "state": "Bihar", "latitude": 25.5941, "longitude": 85.1376, "zone": "Middle Gangetic"},
    {"district": "Ranchi", "state": "Jharkhand", "latitude": 23.3441, "longitude": 85.3096, "zone": "Eastern Plateau"},
    {"district": "Bardhaman", "state": "West Bengal", "latitude": 23.2324, "longitude": 87.8615, "zone": "Lower Gangetic"},
    {"district": "Cuttack", "state": "Odisha", "latitude": 20.4625, "longitude": 85.8828, "zone": "East Coast Plains"},
    {"district": "Bhopal", "state": "Madhya Pradesh", "latitude": 23.2599, "longitude": 77.4126, "zone": "Central Plateau"},
    {"district": "Raipur", "state": "Chhattisgarh", "latitude": 21.2514, "longitude": 81.6296, "zone": "Eastern Plateau"},
    {"district": "Jodhpur", "state": "Rajasthan", "latitude": 26.2389, "longitude": 73.0243, "zone": "Western Dry"},
    {"district": "Anand", "state": "Gujarat", "latitude": 22.5645, "longitude": 72.9289, "zone": "Gujarat Plains"},
    {"district": "Akola", "state": "Maharashtra", "latitude": 20.7002, "longitude": 77.0082, "zone": "Vidarbha"},
    {"district": "Pune", "state": "Maharashtra", "latitude": 18.5204, "longitude": 73.8567, "zone": "Western Plateau"},
    {"district": "Guntur", "state": "Andhra Pradesh", "latitude": 16.3067, "longitude": 80.4365, "zone": "East Coast Plains"},
    {"district": "Warangal", "state": "Telangana", "latitude": 17.9689, "longitude": 79.5941, "zone": "Southern Plateau"},
    {"district": "Dharwad", "state": "Karnataka", "latitude": 15.4589, "longitude": 75.0078, "zone": "Northern Transition"},
    {"district": "Coimbatore", "state": "Tamil Nadu", "latitude": 11.0168, "longitude": 76.9558, "zone": "Southern Plateau"},
    {"district": "Palakkad", "state": "Kerala", "latitude": 10.7867, "longitude": 76.6548, "zone": "West Coast Wet"},
    {"district": "Kamrup", "state": "Assam", "latitude": 26.2006, "longitude": 91.5606, "zone": "Brahmaputra Valley"},
    {"district": "Shillong", "state": "Meghalaya", "latitude": 25.5788, "longitude": 91.8933, "zone": "Eastern Himalayan"},
    {"district": "Dimapur", "state": "Nagaland", "latitude": 25.9068, "longitude": 93.7274, "zone": "Eastern Himalayan"},
    {"district": "Imphal West", "state": "Manipur", "latitude": 24.8170, "longitude": 93.9368, "zone": "Eastern Himalayan"},
    {"district": "Agartala", "state": "Tripura", "latitude": 23.8315, "longitude": 91.2868, "zone": "Eastern Himalayan"},
    {"district": "Gangtok", "state": "Sikkim", "latitude": 27.3389, "longitude": 88.6065, "zone": "Eastern Himalayan"}
]

NASA_POWER_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
PARAMETERS = "T2M_MAX,T2M_MIN,T2M,RH2M,PRECTOTCORR,WS10M,WS10M_MAX,PS,ALLSKY_SFC_SW_DWN,T2MDEW"
START_DATE = "20210101"
END_DATE = "20241231"

def fetch_point(loc):
    params = {
        "parameters": PARAMETERS,
        "community": "AG",
        "longitude": loc["longitude"],
        "latitude": loc["latitude"],
        "start": START_DATE,
        "end": END_DATE,
        "format": "JSON"
    }
    s = requests.Session()
    for attempt in range(4):
        try:
            r = s.get(NASA_POWER_URL, params=params, timeout=35)
            if r.status_code == 200:
                p_data = r.json()["properties"]["parameter"]
                sub = pd.DataFrame(p_data).reset_index().rename(columns={"index": "date_str"})
                sub["date"] = pd.to_datetime(sub["date_str"], format="%Y%m%d")
                sub.drop(columns=["date_str"], inplace=True)
                sub["district"] = loc["district"]
                sub["state"] = loc["state"]
                sub["zone"] = loc["zone"]
                sub["latitude"] = loc["latitude"]
                sub["longitude"] = loc["longitude"]
                return sub
            time.sleep(1.0)
        except Exception:
            time.sleep(1.5)
    return None

print(f"1/3. Extracting 28 Pan-India states via NASA POWER (6 workers)...")
batch_dfs = []
with ThreadPoolExecutor(max_workers=6) as ex:
    futures = {ex.submit(fetch_point, loc): loc for loc in INDIAN_GRID_28}
    for f in tqdm(as_completed(futures), total=len(futures), desc="Fetching Weather"):
        res = f.result()
        if res is not None:
            batch_dfs.append(res)

if not batch_dfs:
    raise RuntimeError("Extraction failed. Please verify your internet connection.")

df_raw = pd.concat(batch_dfs, ignore_index=True)

# Standardize names & units
rename_dict = {
    "T2M": "temperature_2m_mean",
    "T2M_MIN": "temperature_2m_min",
    "T2M_MAX": "temperature_2m_max",
    "RH2M": "relative_humidity_2m_mean",
    "PRECTOTCORR": "precipitation_sum",
    "WS10M": "wind_speed_10m_mean",
    "WS10M_MAX": "wind_speed_10m_max",
    "PS": "surface_pressure_mean",
    "ALLSKY_SFC_SW_DWN": "shortwave_radiation_sum",
    "T2MDEW": "dew_point_2m_mean"
}
df_raw.rename(columns=rename_dict, inplace=True)
df_raw.replace(-999.0, np.nan, inplace=True)

df_raw["wind_speed_10m_max"] = df_raw["wind_speed_10m_max"] * 3.6
df_raw["wind_speed_10m_mean"] = df_raw["wind_speed_10m_mean"] * 3.6
df_raw["shortwave_radiation_sum"] = df_raw["shortwave_radiation_sum"].clip(lower=0.0)
df_raw["precipitation_sum"] = df_raw["precipitation_sum"].clip(lower=0.0)
df_raw["relative_humidity_2m_mean"] = df_raw["relative_humidity_2m_mean"].clip(lower=1.0, upper=100.0)

lead_cols = ["date", "district", "state", "zone", "latitude", "longitude"]
tail_cols = [c for c in df_raw.columns if c not in lead_cols]
df_clean = df_raw[lead_cols + tail_cols].sort_values(by=["district", "date"]).reset_index(drop=True)

# In-memory interpolation
numeric_cols = [c for c in tail_cols if c in df_clean.columns]
df_clean[numeric_cols] = df_clean.groupby("district")[numeric_cols].transform(
    lambda grp: grp.interpolate(method="linear").bfill().ffill()
)

print(f"Extracted {len(df_clean):,} daily observations across {df_clean['district'].nunique()} districts.")

# ------------------------------------------------------------------------------
# 2/3. AGROMET FEATURE ENGINEERING (IN-MEMORY)
# ------------------------------------------------------------------------------
print("2/3. Engineering rolling stress horizons & physical indices...")
df_clean["year"] = df_clean["date"].dt.year
df_clean["month"] = df_clean["date"].dt.month
df_clean["day_of_year"] = df_clean["date"].dt.dayofyear
df_clean["doy_sin"] = np.sin(2 * np.pi * df_clean["day_of_year"] / 365.25)
df_clean["doy_cos"] = np.cos(2 * np.pi * df_clean["day_of_year"] / 365.25)

def assign_season(m):
    return "KHARIF" if m in [6, 7, 8, 9, 10] else ("RABI" if m in [11, 12, 1, 2, 3] else "ZAID")

df_clean["agro_season"] = df_clean["month"].apply(assign_season)

# VPD (kPa)
es = 0.61078 * np.exp((17.27 * df_clean["temperature_2m_mean"]) / (df_clean["temperature_2m_mean"] + 237.3))
ea = es * (df_clean["relative_humidity_2m_mean"] / 100.0)
df_clean["vapor_pressure_deficit"] = np.clip(es - ea, 0, None)

# Hargreaves-Samani ET0
t_diff = np.clip(df_clean["temperature_2m_max"] - df_clean["temperature_2m_min"], 0.1, None)
df_clean["et0_fao_evapotranspiration"] = (
    0.0023 * (df_clean["temperature_2m_mean"] + 17.8) * np.sqrt(t_diff) * (df_clean["shortwave_radiation_sum"] * 0.408)
).clip(lower=0.5, upper=12.0)

# Grouped Rolling Aggregations
grp = df_clean.groupby("district")
df_clean["rainfall_1d"] = df_clean["precipitation_sum"]
df_clean["rainfall_3d_sum"] = grp["precipitation_sum"].transform(lambda x: x.rolling(3, min_periods=1).sum())
df_clean["rainfall_7d_sum"] = grp["precipitation_sum"].transform(lambda x: x.rolling(7, min_periods=1).sum())
df_clean["rainfall_14d_sum"] = grp["precipitation_sum"].transform(lambda x: x.rolling(14, min_periods=1).sum())

df_clean["temperature_3d_mean"] = grp["temperature_2m_mean"].transform(lambda x: x.rolling(3, min_periods=1).mean())
df_clean["temperature_7d_mean"] = grp["temperature_2m_mean"].transform(lambda x: x.rolling(7, min_periods=1).mean())
df_clean["temperature_3d_max"] = grp["temperature_2m_max"].transform(lambda x: x.rolling(3, min_periods=1).max())
df_clean["temperature_7d_max"] = grp["temperature_2m_max"].transform(lambda x: x.rolling(7, min_periods=1).max())
df_clean["temperature_3d_min"] = grp["temperature_2m_min"].transform(lambda x: x.rolling(3, min_periods=1).min())

df_clean["humidity_3d_mean"] = grp["relative_humidity_2m_mean"].transform(lambda x: x.rolling(3, min_periods=1).mean())
df_clean["humidity_7d_mean"] = grp["relative_humidity_2m_mean"].transform(lambda x: x.rolling(7, min_periods=1).mean())
df_clean["et0_3d_sum"] = grp["et0_fao_evapotranspiration"].transform(lambda x: x.rolling(3, min_periods=1).sum())
df_clean["et0_7d_sum"] = grp["et0_fao_evapotranspiration"].transform(lambda x: x.rolling(7, min_periods=1).sum())
df_clean["wind_speed_3d_mean"] = grp["wind_speed_10m_max"].transform(lambda x: x.rolling(3, min_periods=1).mean())

def calc_streaks(series, condition_fn):
    flags = condition_fn(series).astype(int)
    streaks, cur = [], 0
    for v in flags:
        cur = cur + 1 if v == 1 else 0
        streaks.append(cur)
    return pd.Series(streaks, index=series.index)

df_clean["consecutive_dry_days"] = grp["precipitation_sum"].transform(lambda s: calc_streaks(s, lambda x: x < 1.0))
df_clean["consecutive_rain_days"] = grp["precipitation_sum"].transform(lambda s: calc_streaks(s, lambda x: x >= 2.5))

# ------------------------------------------------------------------------------
# 3/3. ICAR AGRONOMIC ALIGNMENT & TARGET FORMULATION
# ------------------------------------------------------------------------------
print("3/3. Formulating ICAR stage-specific target labels...")
ICAR_CROP_MATRIX = {
    "Wheat": {
        "season": "RABI",
        "stages": ["Vegetative", "Flowering", "Grain_Filling", "Maturity"],
        "temp_opt_min": 12.0, "temp_opt_max": 25.0, "heat_limit": 32.0,
        "water_need_7d": 25.0
    },
    "Rice": {
        "season": "KHARIF",
        "stages": ["Transplanting", "Tillering", "Flowering", "Maturity"],
        "temp_opt_min": 20.0, "temp_opt_max": 33.0, "heat_limit": 38.0,
        "water_need_7d": 45.0
    },
    "Cotton": {
        "season": "KHARIF",
        "stages": ["Vegetative", "Square_Formation", "Boll_Formation", "Maturity"],
        "temp_opt_min": 21.0, "temp_opt_max": 32.0, "heat_limit": 39.0,
        "water_need_7d": 28.0
    },
    "Maize": {
        "season": "KHARIF",
        "stages": ["Vegetative", "Tasseling", "Silking", "Maturity"],
        "temp_opt_min": 18.0, "temp_opt_max": 30.0, "heat_limit": 36.0,
        "water_need_7d": 30.0
    },
    "Mustard": {
        "season": "RABI",
        "stages": ["Vegetative", "Flowering", "Pod_Formation", "Maturity"],
        "temp_opt_min": 10.0, "temp_opt_max": 22.0, "heat_limit": 30.0,
        "water_need_7d": 18.0
    },
    "Soybean": {
        "season": "KHARIF",
        "stages": ["Vegetative", "Pod_Initiation", "Pod_Filling", "Maturity"],
        "temp_opt_min": 20.0, "temp_opt_max": 30.0, "heat_limit": 37.0,
        "water_need_7d": 28.0
    }
}

aligned = []
for crop_name, p in ICAR_CROP_MATRIX.items():
    sub = df_clean[df_clean["agro_season"] == p["season"]].copy()
    sub["crop"] = crop_name

    if p["season"] == "RABI":
        s_map = {11: p["stages"][0], 12: p["stages"][1], 1: p["stages"][1], 2: p["stages"][2], 3: p["stages"][3]}
    else:
        s_map = {6: p["stages"][0], 7: p["stages"][0], 8: p["stages"][1], 9: p["stages"][2], 10: p["stages"][3]}

    sub["crop_stage"] = sub["month"].map(s_map).fillna(p["stages"][0])
    sub["temp_opt_min"] = p["temp_opt_min"]
    sub["temp_opt_max"] = p["temp_opt_max"]
    sub["heat_limit"] = p["heat_limit"]
    sub["water_need_7d"] = p["water_need_7d"]
    aligned.append(sub)

df_ml = pd.concat(aligned, ignore_index=True)

# TARGET 1: Heat Stress
c_heat = [
    (df_ml["temperature_3d_max"] >= df_ml["heat_limit"] + 2.0),
    (df_ml["temperature_3d_max"] >= df_ml["heat_limit"])
]
df_ml["heat_stress"] = np.select(c_heat, ["HIGH", "MODERATE"], default="LOW")

# TARGET 2: Waterlogging Risk
c_waterlog = [
    (df_ml["rainfall_3d_sum"] > 110.0) & (df_ml["crop"] != "Rice"),
    (df_ml["rainfall_3d_sum"] > 65.0) & (df_ml["crop"] != "Rice")
]
df_ml["waterlogging_risk"] = np.select(c_waterlog, ["HIGH", "MODERATE"], default="LOW")

# TARGET 3: Irrigation Requirement
water_deficit = df_ml["et0_7d_sum"] - df_ml["rainfall_7d_sum"]
c_irrig = [
    (water_deficit > 35.0) & (df_ml["rainfall_3d_sum"] < 2.0),
    (water_deficit > 20.0) & (df_ml["rainfall_3d_sum"] < 8.0),
    (water_deficit > 10.0) & (df_ml["rainfall_3d_sum"] < 15.0)
]
df_ml["irrigation"] = np.select(
    c_irrig,
    ["URGENT_IRRIGATION", "IRRIGATION_REQUIRED", "LIGHT_IRRIGATION"],
    default="NO_IRRIGATION"
)
df_ml.loc[df_ml["rainfall_3d_sum"] >= 20.0, "irrigation"] = "NO_IRRIGATION"

# TARGET 4: Spraying Suitability (CIBRC / ASABE Standards)
cond_spray_unsafe = (
    (df_ml["wind_speed_10m_max"] > 16.0) |
    (df_ml["precipitation_sum"] > 1.0) |
    (df_ml["temperature_2m_max"] > 35.0)
)
cond_spray_wait = (
    (df_ml["wind_speed_10m_max"] < 3.0) |
    (df_ml["relative_humidity_2m_mean"] > 85.0) |
    (df_ml["relative_humidity_2m_mean"] < 35.0)
)
df_ml["spraying_suitability"] = np.where(
    cond_spray_unsafe, "UNSAFE",
    np.where(cond_spray_wait, "WAIT", "SUITABLE")
)

# TARGET 5: Composite Crop Weather Risk
pts = (
    (df_ml["heat_stress"] == "HIGH").astype(int) * 3 +
    (df_ml["heat_stress"] == "MODERATE").astype(int) * 1 +
    (df_ml["waterlogging_risk"] == "HIGH").astype(int) * 3 +
    (df_ml["waterlogging_risk"] == "MODERATE").astype(int) * 1 +
    (df_ml["irrigation"] == "URGENT_IRRIGATION").astype(int) * 2 +
    (df_ml["consecutive_dry_days"] > 21).astype(int) * 2
)
df_ml["crop_weather_risk"] = np.select(
    [pts >= 5, pts >= 3, pts >= 1],
    ["SEVERE", "HIGH", "MODERATE"],
    default="LOW"
)

# Persist both to disk AND keep active in-memory variable `df_ml`
df_ml.to_parquet("pan_india_ml_ready.parquet", engine="pyarrow", index=False)

print("\n" + "=" * 60)
print(f"PIPELINE COMPLETED SUCCESSFULLY!")
print(f"Total ML Training Samples: {len(df_ml):,} rows x {df_ml.shape[1]} columns")
print("In-memory variable `df_ml` is ready for model training.")
print("=" * 60)

C:\Users\ksudi\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1/3. Extracting 28 Pan-India states via NASA POWER (6 workers)...


Fetching Weather:   0%|          | 0/28 [00:00<?, ?it/s]

Fetching Weather:   4%|▎         | 1/28 [00:03<01:29,  3.31s/it]

Fetching Weather:  11%|█         | 3/28 [00:03<00:22,  1.11it/s]

...

Fetching Weather: 100%|██████████| 28/28 [00:14<00:00,  2.44it/s]

Fetching Weather: 100%|██████████| 28/28 [00:14<00:00,  1.89it/s]

Extracted 40,908 daily observations across 28 districts.
2/3. Engineering rolling stress horizons & physical indices...


3/3. Formulating ICAR stage-specific target labels...



PIPELINE COMPLETED SUCCESSFULLY!
Total ML Training Samples: 102,424 rows x 51 columns
In-memory variable `df_ml` is ready for model training.


In [2]:
# ==============================================================================
# CELL 1: ROBUST RE-TRAIN WITH EXACT FEATURE CONTRACTS
# ==============================================================================
import joblib
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

# 1. Exact list of 28 numeric features + 3 encoded categorical features
CORE_NUMERIC_FEATURES = [
    "latitude", "longitude",
    "temperature_2m_mean", "temperature_2m_min", "temperature_2m_max",
    "relative_humidity_2m_mean", "precipitation_sum", "wind_speed_10m_max",
    "surface_pressure_mean", "shortwave_radiation_sum", "dew_point_2m_mean",
    "vapor_pressure_deficit", "et0_fao_evapotranspiration",
    "rainfall_1d", "rainfall_3d_sum", "rainfall_7d_sum", "rainfall_14d_sum",
    "temperature_3d_mean", "temperature_7d_mean", "temperature_3d_max",
    "temperature_7d_max", "temperature_3d_min",
    "humidity_3d_mean", "humidity_7d_mean", "et0_3d_sum", "et0_7d_sum",
    "wind_speed_3d_mean", "consecutive_dry_days", "consecutive_rain_days",
    "doy_sin", "doy_cos", "month"
]

CATEGORICAL_COLS = ["crop", "crop_stage", "agro_season"]

# Encode categoricals in df_ml
cat_encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df_ml[col + "_enc"] = le.fit_transform(df_ml[col].astype(str))
    cat_encoders[col] = le

ALL_MODEL_FEATURES = CORE_NUMERIC_FEATURES + [c + "_enc" for c in CATEGORICAL_COLS]

# Train/Test split
train_mask = df_ml["year"] <= 2023
test_mask = df_ml["year"] == 2024

X_train = df_ml.loc[train_mask, ALL_MODEL_FEATURES].astype(np.float32)
X_test = df_ml.loc[test_mask, ALL_MODEL_FEATURES].astype(np.float32)

TARGETS = [
    "crop_weather_risk",
    "irrigation",
    "heat_stress",
    "waterlogging_risk",
    "spraying_suitability"
]

trained_models = {}
target_encoders = {}

print("Training production models with strict feature locking...")
for tgt in TARGETS:
    t_le = LabelEncoder()
    y_tr = t_le.fit_transform(df_ml.loc[train_mask, tgt])
    target_encoders[tgt] = t_le

    clf = lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.08,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    clf.fit(X_train, y_tr)
    trained_models[tgt] = clf
    print(f"✓ Model trained for {tgt} (Classes: {list(t_le.classes_)})")

# Save the exact metadata and ordering
pipeline_bundle = {
    "models": trained_models,
    "target_encoders": target_encoders,
    "cat_encoders": cat_encoders,
    "numeric_features": CORE_NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_COLS,
    "all_features": ALL_MODEL_FEATURES
}

joblib.dump(pipeline_bundle, "farmer_advisory_pipeline.pkl")
print("\n" + "=" * 60)
print("SUCCESS: Pipeline trained & saved to farmer_advisory_pipeline.pkl")
print("=" * 60)

Training production models with strict feature locking...


✓ Model trained for crop_weather_risk (Classes: ['HIGH', 'LOW', 'MODERATE', 'SEVERE'])


✓ Model trained for irrigation (Classes: ['IRRIGATION_REQUIRED', 'LIGHT_IRRIGATION', 'NO_IRRIGATION', 'URGENT_IRRIGATION'])


✓ Model trained for heat_stress (Classes: ['HIGH', 'LOW', 'MODERATE'])


✓ Model trained for waterlogging_risk (Classes: ['HIGH', 'LOW', 'MODERATE'])


✓ Model trained for spraying_suitability (Classes: ['SUITABLE', 'UNSAFE', 'WAIT'])

SUCCESS: Pipeline trained & saved to farmer_advisory_pipeline.pkl


In [3]:
# ==============================================================================
# FINAL REPO EXPORT
# ==============================================================================
from pathlib import Path
import json
import joblib

OUT_DIR = Path("models") / "agriculture"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = OUT_DIR / "agriculture.pkl"
META_PATH = OUT_DIR / "metadata.json"

# `pipeline_bundle` is created by the preceding source-faithful training cell.
joblib.dump(pipeline_bundle, MODEL_PATH)

meta = {
    "artifact": "agriculture.pkl",
    "kind": "farmer_advisory_multitarget_lightgbm_bundle",
    "task": "multi_target_classification",
    "numeric_features": list(CORE_NUMERIC_FEATURES),
    "categorical_features": list(CATEGORICAL_COLS),
    "features": list(ALL_MODEL_FEATURES),
    "n_features": int(len(ALL_MODEL_FEATURES)),
    "targets": list(TARGETS),
    "training_data_source": "NASA POWER daily point API",
    "training_period": ["2021-01-01", "2024-12-31"],
    "training_anchor_locations": 28,
    "notes": [
        "The notebook creates rule-derived agronomic targets.",
        "Final production models are LightGBM LGBMClassifier instances.",
        "Categorical values are LabelEncoded and saved inside the bundle."
    ]
}

META_PATH.write_text(json.dumps(meta, indent=2), encoding="utf-8")

print("Agriculture model exported:", MODEL_PATH)
print("Metadata exported:", META_PATH)


Agriculture model exported: models\agriculture\agriculture.pkl
Metadata exported: models\agriculture\metadata.json
